# Geographic Classification of UK Electric Vehicle Infrastructure
**Module:** Data Mining (CSO7021)  
**Student ID:** 2418010  
**Artifact:** Final Project Replication Notebook  

---

## Project Context & Business Problem
The statutory transition in the United Kingdom from Internal Combustion Engine (ICE) vehicles to Electric Vehicles (EVs) presents a significant geographic challenge, namely, infrastructure disparity. Urban centres and affluent regions are more likely to attract robust public charging systems, whereas rural or economically distinct local authorities may experience limited access.

Addressing the risk of localised 'infrastructure deserts' requires policy-makers and energy providers to understand the socioeconomic factors influencing charging station deployment.

### The Core Data Mining Question
> **"Can we accurately predict whether a UK Local Authority area is a 'High' or 'Low' public EV charging infrastructure density zone based entirely on its economic and demographic profiles?"**

By framing this as a binary classification problem, the analysis seeks to determine whether underlying regional characteristics, such as individual disposable income and population metrics, serve as strong leading indicators of public infrastructure investment.

---

## Methodology & Boundary Constraints
This notebook presents a complete and reproducible data mining pipeline, encompassing all stages from raw data ingestion to model evaluation. The methodology strictly adheres to the module assessment criteria:
* All preprocessing, feature engineering, and modelling techniques are limited to the tools and methods introduced during Weeks 1 to 6 of the module.
* The feature space is constructed using official government statistics obtained from the **Department for Transport (DfT)** and the **Office for National Statistics (ONS)**.
* The pipeline is fully self-contained, enabling evaluators to execute the artefact in its entirety without requiring external file modifications.

---

## 1. Data Loading and Inspection
We begin by reading the modular, extracted raw files from the directory.

### 1.1 Ingestion of Heterogeneous Administrative Frameworks

In [5]:
import pandas as pd
import numpy as np

# Define relative paths for local execution to ensure marker reproducibility
ev_data_path = "dft_ev_charging_raw.xlsx"
wealth_data_path = "ons_gdhi_per_head_raw.xlsx"
pop_data_path = "ons_population_raw.xlsx"

# Load the raw files directly into separate DataFrames
df_ev_raw = pd.read_excel(ev_data_path)
df_wealth_raw = pd.read_excel(wealth_data_path)
df_pop_raw = pd.read_excel(pop_data_path)

# Verify foundational matrix dimensions before structural manipulation
print("--- RAW INGESTION DIMENSIONS ---")
print(f"DfT EV Infrastructure Raw Shape : {df_ev_raw.shape}")
print(f"ONS GDHI Wealth Raw Shape       : {df_wealth_raw.shape}")
print(f"ONS Population Raw Shape        : {df_pop_raw.shape}")

--- RAW INGESTION DIMENSIONS ---
DfT EV Infrastructure Raw Shape : (363, 6)
ONS GDHI Wealth Raw Shape       : (362, 30)
ONS Population Raw Shape        : (362, 30)


#### Technical Evaluation: Raw Ingestion Sizing
Administrative Baseline Secured: Initial data loading successfully imports all three administrative files. The matching horizontal dimensions (362 and 363 rows) indicate that the datasets cover a structurally consistent list of geographic boundaries across both the ONS and DfT tracking systems.

Schema Volume Bounds: The ONS files provide a 30-column feature matrix, reflecting a multi-decade timeline of socioeconomic attributes. The raw structural format matches the demands of a multi-variable classification baseline, though explicit row filtering must follow to remove non-data records.

### 1.2 Administrative Metadata Stripping and Header Alignment

In [6]:
# Strip decorative text block headers to isolate active data tables
# DfT EV infrastructure requires a row 1 offset index
df_ev = df_ev_raw.copy()
df_ev.columns = df_ev.iloc[1].values
df_ev = df_ev.iloc[2:].reset_index(drop=True)

# ONS Wealth requires a row 0 offset index
df_wealth = df_wealth_raw.copy()
df_wealth.columns = df_wealth.iloc[0].values
df_wealth = df_wealth.iloc[1:].reset_index(drop=True)

# ONS Population requires a row 0 offset index
df_pop = df_pop_raw.copy()
df_pop.columns = df_pop.iloc[0].values
df_pop = df_pop.iloc[1:].reset_index(drop=True)

print("--- POST-RECONSTRUCTION SHAPES ---")
print(f"Cleaned EV Matrix Shape     : {df_ev.shape}")
print(f"Cleaned Wealth Matrix Shape : {df_wealth.shape}")
print(f"Cleaned Pop Matrix Shape    : {df_pop.shape}")

--- POST-RECONSTRUCTION SHAPES ---
Cleaned EV Matrix Shape     : (361, 6)
Cleaned Wealth Matrix Shape : (361, 30)
Cleaned Pop Matrix Shape    : (361, 30)


#### Technical Evaluation: Slicing Verification and Row Synchronization
Administrative Metadata Successfully Stripped: Automated header extraction via explicit row-index offsets (iloc) has successfully trimmed decorative title lines and empty spacer blocks from all frames.

Granularity Harmonization: The post-reconstruction shapes show a perfect alignment of exactly 361 rows across the EV, Wealth, and Population matrices. This confirms that all three datasets have been isolated at the local authority district (LAD) level, providing an un-aggregated foundation that preserves real geographic variance for machine learning modeling.

1.3 Baseline Data Type and Schema Verification

In [7]:
print("--- EV COLUMNS ---")
print(df_ev.columns.tolist()[:10])

print("\n--- WEALTH COLUMNS ---")
print(df_wealth.columns.tolist()[:10])

print("\n--- POPULATION COLUMNS ---")
print(df_pop.columns.tolist()[:10])

--- EV COLUMNS ---
['Local authority code', 'Local authority', 'County code', 'County', '1 January 2026', '1 April 2026']

--- WEALTH COLUMNS ---
['Region', 'LAD code', 'Region name', np.int64(1997), np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003)]

--- POPULATION COLUMNS ---
['Region', 'LAD code', 'Region name', np.int64(1997), np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003)]


#### Technical Evaluation: Feature Schema Mapping
Target Feature Identification: The DfT schema confirms the presence of explicit, side-by-side snapshot variables (`1 January 2026` and `1 April 2026`). The 1 April 2026 parameter will serve as our primary modeling target.

Temporal Typing Consistency: The ONS annual features are correctly mapped as `np.int64` numerical column names rather than generic text strings. Because both ONS dataframes share identical year names, explicit tracking suffixes (`_wealth` and `_pop`) must be handled programmatically during the upcoming merge to avoid namespace overlap.

## 2. Preprocessing

### 2.1: Key Standardization and Boundary Realignment

In [8]:
# Strip trailing or leading spaces from geographic keys to ensure join consistency
df_wealth['LAD code'] = df_wealth['LAD code'].astype(str).str.strip()
df_pop['LAD code'] = df_pop['LAD code'].astype(str).str.strip()
df_ev['Local authority code'] = df_ev['Local authority code'].astype(str).str.strip()

# 2. Programmatically resolve the 2025 boundary tracking misalignment
# Map up-to-date DfT tracking codes back to their historical ONS equivalents
df_ev_aligned = df_ev.copy()
df_ev_aligned['Local authority code'] = df_ev_aligned['Local authority code'].replace({
    'E08000038': 'E08000016',  # Map modern Barnsley to ONS legacy code
    'E08000039': 'E08000019'   # Map modern Sheffield to ONS legacy code
})

# 3. Filter out rows that lack valid Local Authority formats (e.g. administrative strings or missing flags)
df_ev_aligned = df_ev_aligned[df_ev_aligned['Local authority code'].str.match(r'^(E0|W0|S1|N0)', na=False)]

# 4. Verify the distinct key intersections across the frameworks
ons_codes_set = set(df_wealth['LAD code'].dropna().unique())
ev_codes_set = set(df_ev_aligned['Local authority code'].dropna().unique())
matching_keys = ons_codes_set.intersection(ev_codes_set)

print("--- GEOGRAPHIC PROFILE SYNCHRONIZATION ---")
print(f"Unique ONS Geographic Keys: {len(ons_codes_set)}")
print(f"Unique DfT Geographic Keys: {len(ev_codes_set)}")
print(f"Relational Keys Intersected : {len(matching_keys)} / 361")

--- GEOGRAPHIC PROFILE SYNCHRONIZATION ---
Unique ONS Geographic Keys: 361
Unique DfT Geographic Keys: 361
Relational Keys Intersected : 361 / 361
